# МТС · Qwen3-0.6B · длительное продолжение last128

**2863 новых train / 366 inner-dev / 13536 обучающих фактов.** Один проход, максимум 716 обновлений. Бюджет цикла до 4 часов; загрузка и оценка дополнительно. Все даты v2 участвуют в loss. Данные/тексты не обрезаются; контекст 24576. Адаптер last128 уже внутри payload. Запускай в отдельном Colab, не останавливая 0.8B. Обучение выключено до явного RUN_TRAINING=True.


## 1. GPU и внутренний payload

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib,json,shutil,subprocess,sys,zipfile
import torch
from google.colab import files
assert torch.cuda.is_available(),'Выберите среду Colab с GPU'
print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
print('Torch:',torch.__version__,'CUDA:',torch.version.cuda)
"""Notebook upload helper: content identity, repeatable verification, no overwrite."""
from pathlib import Path
import hashlib
import io
import json
import zipfile


def prepare_payload(root, uploaded, expected_zip_sha, expected_manifest_sha):
    root = Path(root).resolve()
    if not root.exists() or not any(root.iterdir()):
        matches = [b for b in (uploaded or {}).values()
                   if hashlib.sha256(b).hexdigest() == expected_zip_sha]
        if not matches:
            raise ValueError('Среди загруженных файлов нет выданного payload: контрольная сумма отличается')
        with zipfile.ZipFile(io.BytesIO(matches[0])) as z:
            if z.testzip() is not None or sum(i.file_size for i in z.infolist()) >= 256 * 1024**2:
                raise ValueError('Повреждённый или слишком большой payload')
            for item in z.infolist():
                path = (root.parent / item.filename).resolve()
                if not path.is_relative_to(root) or (item.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError('Недопустимый путь в payload')
            z.extractall(root.parent)
    manifest_path = root/'manifest.json'
    if not manifest_path.is_file() or hashlib.sha256(manifest_path.read_bytes()).hexdigest() != expected_manifest_sha:
        raise ValueError('Manifest отсутствует или изменён. Папка не перезаписана; сохраните результаты и восстановите payload в новой среде')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    for rel, expected in manifest['sha256'].items():
        path = (root/rel).resolve()
        if not path.is_relative_to(root) or not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise ValueError('Файл отсутствует или изменён, папка не перезаписана: '+rel)
    return manifest

ROOT=Path('/content/mts_qwen06_extended')
ZIP_SHA='392e66072d40bdbabcffddf833e3f018d04ab89a584cfe5519de62d08f45f6b3'
MANIFEST_SHA='105a9b2aefcaa2e9a13e0802feac82b2281ffbf958446270952a0079f6bb4383'
if ROOT.exists() and any(ROOT.iterdir()):
    manifest=prepare_payload(ROOT,None,ZIP_SHA,MANIFEST_SHA)
else:
    uploaded=files.upload()
    manifest=prepare_payload(ROOT,uploaded,ZIP_SHA,MANIFEST_SHA)
    del uploaded
print('Payload проверен:',manifest['counts'])


## 2. Зависимости и проверка всех 3229 примеров — без обучения


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'requirements.txt')],check=True)
subprocess.run([sys.executable,str(ROOT/'verify_extended.py'),'--tokenizer'],cwd=ROOT,check=True)


## 3. Drive и параметры

Для нового запуска оставь RESUME_CHECKPOINT пустым. Для восстановления после обрыва укажи полный путь из last_pointer.json прежнего RUN. При возобновлении нужен тот же payload и LR. Эта ячейка создаёт новое имя сегмента. Drive подключается вручную.


In [ ]:
RESUME_CHECKPOINT = ''
SAVE_TO_DRIVE = True
LEARNING_RATE = 0.00005
BUDGET_HOURS = 4
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUNS=Path('/content/drive/MyDrive/mts_qwen06_extended_runs')
else:
    RUNS=Path('/content/mts_qwen06_extended_runs')
RUN=RUNS/datetime.now(timezone.utc).strftime('extended_%Y%m%dT%H%M%SZ')
if RESUME_CHECKPOINT: assert (Path(RESUME_CHECKPOINT)/'snapshot_hashes.json').is_file()
print('Папка результатов:',RUN,'Бюджет цикла:',BUDGET_HOURS,'ч')


## 4. Единственный запуск

Поставь RUN_TRAINING=True. Вначале — проверки loss/длинного звонка и оценка до обучения на 32 dev; это ещё не обновления весов. Дождись TRAINING_ESTIMATE после трёх успешных обновлений. Ранняя диагностика — шаг16. Last сохраняется после16, каждые64, после96 и на последнем шаге. Качество выбирается на32 звонках после96,384 и в конце. Best и last проверяются отдельно, если различаются. Полная инструкция — START_HERE.md.


In [ ]:
RUN_TRAINING = False

def download_result():
    if not RUN.exists():
        print('Результатов пока нет');return
    console=RUN.parent/(RUN.name+'.console.log')
    if console.exists(): shutil.copyfile(console,RUN/'console.log')
    with zipfile.ZipFile(RUN/'executed_source.zip','w',zipfile.ZIP_DEFLATED) as z:
        for rel in manifest['sha256']:
            if rel.endswith('.py') or rel in ('requirements.txt','selection.json','data_policy.json'):
                z.write(ROOT/rel,rel)
        z.write(ROOT/'manifest.json','payload_manifest.json')
    keep=set()
    for name in ('last','best'):
        p=RUN/(name+'_pointer.json')
        if p.exists(): keep.add(Path(json.loads(p.read_text())['relative_path']).as_posix())
    archive=Path('/content')/(RUN.name+'_results.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in RUN.rglob('*'):
            if not p.is_file():continue
            rel=p.relative_to(RUN)
            if rel.parts[0]=='checkpoints' and Path(*rel.parts[:2]).as_posix() not in keep:continue
            z.write(p,rel.as_posix())
    print('Архив с best/last и состояниями optimizer:',archive)
    files.download(str(archive))

if RUN_TRAINING:
    assert not RUN.exists(),'Создайте новую папку через раздел 3; старую не перезаписываем'
    RUN.parent.mkdir(parents=True,exist_ok=True)
    command=[sys.executable,'-u','-m','ml.qwen.extended_qwen06','--root',str(ROOT),'--out',str(RUN),
             '--precision','float16','--learning-rate',str(LEARNING_RATE),'--budget-hours',str(BUDGET_HOURS)]
    if RESUME_CHECKPOINT:command+=['--resume',RESUME_CHECKPOINT]
    process=subprocess.Popen(command,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
        text=True,encoding='utf-8',errors='replace',bufsize=1)
    try:
        with (RUN.parent/(RUN.name+'.console.log')).open('w',encoding='utf-8') as log:
            for line in process.stdout:
                log.write(line);log.flush();print(line,end='',flush=True)
        result=process.wait()
        if result:raise subprocess.CalledProcessError(result,command)
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:process.wait(timeout=60)
            except subprocess.TimeoutExpired:process.kill();process.wait()
        raise
    finally:download_result()
else:print('Обучение выключено. Для запуска установите RUN_TRAINING=True.')


## 5. Результаты и ошибки

In [ ]:
for name in ('experiment_report.json','candidate_comparison.json','failure.json','last_pointer.json','best_pointer.json'):
    p=RUN/name
    if p.exists():print(name,p.read_text(encoding='utf-8'))
for phase in ('after_dev','after_last_dev32'):
    if (RUN/phase/'predictions.jsonl').exists():
        subprocess.run([sys.executable,'-m','ml.qwen.analyze_results','--gold',str(ROOT/'data/eval32.jsonl'),
            '--prediction-dir',str(RUN/phase),'--evaluator',str(ROOT/'evaluation/case_metrics.py'),
            '--output',str(RUN/(phase+'_error_analysis.json'))],cwd=ROOT,check=True)


## 6. Скачать обновлённый архив

In [ ]:
download_result()


## 7. Необязательная проверка last или всех 366 inner-dev — без обучения

По умолчанию выключена. Позволяет отдельно проверить last, даже если best остался step0. Полный dev может занять часы. Официальных 383 val в этом пакете нет.


In [ ]:
RUN_RECHECK = False
ADAPTER_TO_CHECK = 'last'
EVAL_ALL_INNER_DEV = False
if RUN_RECHECK:
    command=[sys.executable,'-u','-m','ml.qwen.extended_qwen06','--root',str(ROOT),'--out',str(RUN),
             '--precision','float16','--recheck',ADAPTER_TO_CHECK]
    if EVAL_ALL_INNER_DEV:command+=['--eval-all-dev']
    subprocess.run(command,cwd=ROOT,check=True)
else:print('Дополнительная оценка не запускалась')
